# 07 — Threshold Tuning and Freeze

Use the configuration selected by Notebook 06. Refit L1 on TRAIN only, refit the selected Logistic Regression on TRAIN only, tune the classification threshold on VALIDATION only, and freeze the complete inference schema.

The frozen model is Logistic Regression only. No test data is read.

In [1]:
from pathlib import Path
import sys, json, joblib, gc
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
ROOT=Path.cwd()
while ROOT!=ROOT.parent and not (ROOT/"data").exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
TRAIN_CSV=ROOT/"data/processed/train_combined_features.csv"; VAL_CSV=ROOT/"data/processed/val_combined_features.csv"
assert TRAIN_CSV.exists() and VAL_CSV.exists()
from src.features import ALL_COLS
from src.modeling import evaluate
tr=pd.read_csv(TRAIN_CSV,usecols=list(ALL_COLS)+["label"]); va=pd.read_csv(VAL_CSV,usecols=list(ALL_COLS)+["label"])
r=pd.read_csv(ROOT/"metrics/l1_model_comparison.csv").sort_values(["roc_auc","pr_auc"],ascending=False).iloc[0]
cols=list(ALL_COLS); l1_C=float(r.l1_C); model_C=float(r.model_C)
Xtr=tr[cols].to_numpy(dtype=np.float32,copy=True); Xv=va[cols].to_numpy(dtype=np.float32,copy=True)
ytr=tr.label.to_numpy(dtype=np.int8,copy=False)
selector=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("l1",LogisticRegression(penalty="l1",solver="liblinear",C=l1_C,max_iter=3000,class_weight="balanced",random_state=42))])
selector.fit(Xtr,ytr)
mask=selector.named_steps["l1"].coef_[0]!=0; selected=[c for c,m in zip(cols,mask) if m]
idx=[cols.index(c) for c in selected]
Xts=np.ascontiguousarray(Xtr[:,idx],dtype=np.float32); Xvs=np.ascontiguousarray(Xv[:,idx],dtype=np.float32)
model=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("model",LogisticRegression(solver="liblinear",C=model_C,max_iter=3000,class_weight="balanced",random_state=42))])
model.fit(Xts,ytr); p=model.predict_proba(Xvs)[:,1]
scan=[]
for th in np.arange(.01,1.0,.01): scan.append({"threshold":float(th),**evaluate(va.label,p,float(th))})
scan=pd.DataFrame(scan).sort_values(["f1","balanced_accuracy"],ascending=False).reset_index(drop=True); threshold=float(scan.iloc[0].threshold)
models_dir=ROOT/"models"; models_dir.mkdir(exist_ok=True)
selector_path=models_dir/"l1_selector_frozen.joblib"; model_path=models_dir/"model_frozen.joblib"; schema_path=models_dir/"feature_schema_l1.json"
joblib.dump(selector,selector_path,compress=3); joblib.dump(model,model_path,compress=3)
schema={"candidate_feature_set":"all_forensic","l1_C":l1_C,"model_C":model_C,"selected_features":selected,"model":"logistic_regression","threshold":threshold,"training_rows":int(len(tr)),"validation_rows":int(len(va))}
schema_path.write_text(json.dumps(schema,indent=2),encoding="utf-8"); scan.to_csv(ROOT/"metrics/threshold_scan_l1.csv",index=False)
print("FROZEN SCHEMA:"); print(json.dumps(schema,indent=2)); print("\nThreshold candidates:"); print(scan.head(10).to_string(index=False))
print("\nFrozen model size MB:",round(model_path.stat().st_size/1024**2,4)); print("Frozen selector size MB:",round(selector_path.stat().st_size/1024**2,4))
print("Saved:",model_path); print("Saved:",selector_path); print("Saved:",schema_path)


FROZEN SCHEMA:
{
  "candidate_feature_set": "all_forensic",
  "l1_C": 0.1,
  "model_C": 10.0,
  "selected_features": [
    "pixel_mean",
    "pixel_std",
    "pixel_variance",
    "pixel_energy",
    "pixel_mad",
    "pixel_skewness",
    "pixel_kurtosis",
    "gradient_mean",
    "gradient_std",
    "gradient_energy",
    "laplacian_mean",
    "laplacian_std",
    "laplacian_energy",
    "noise_hf_mean",
    "noise_hf_std",
    "noise_hf_energy",
    "LL3_mean",
    "LL3_std",
    "LL3_variance",
    "LL3_mad",
    "LL3_skewness",
    "LL3_kurtosis",
    "LH3_mean",
    "LH3_std",
    "LH3_energy",
    "LH3_mad",
    "LH3_skewness",
    "LH3_kurtosis",
    "HL3_mean",
    "HL3_std",
    "HL3_variance",
    "HL3_energy",
    "HL3_mad",
    "HL3_skewness",
    "HL3_kurtosis",
    "HH3_mean",
    "HH3_std",
    "HH3_variance",
    "HH3_energy",
    "HH3_mad",
    "HH3_skewness",
    "HH3_kurtosis",
    "LH2_mean",
    "LH2_std",
    "LH2_variance",
    "LH2_energy",
    "LH2_mad",
    "L